In [2]:
path = r"C:\Users\HP\Desktop\Ejust\Projects\Data Security\nids\data\9bafce9d380588c2_MOHANAD_A4706\data\NF-ToN-IoT-v2.csv"

In [3]:
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
from pathlib import Path

is_benign = np.concatenate([
    (c["Attack"] == "Benign").to_numpy()
    for c in pd.read_csv(path, usecols=["Attack"], chunksize=500_000)
])
n = len(is_benign)
cum = np.cumsum(is_benign)
total_b = cum[-1]

def row_at(p):
    return int(np.searchsorted(cum, p * total_b))

names = ["train", "val", "tune", "test"]
cuts = [0, row_at(0.60), row_at(0.70), row_at(0.80), n]
GAP = 100_000
bounds = {}
for i, name in enumerate(names):
    start = cuts[i] + (GAP if i > 0 else 0)
    bounds[name] = (start, cuts[i + 1])
    print(name, bounds[name], "rows:", cuts[i + 1] - start,
          "benign:", int(is_benign[start:cuts[i + 1]].sum()))

train (0, 6253698) rows: 6253698 benign: 3659681
val (6353698, 7663487) rows: 1309789 benign: 539110
tune (7763487, 8658262) rows: 894775 benign: 510816
test (8758262, 16940496) rows: 8182234 benign: 1130384


In [4]:
str_cols = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"]
header = pd.read_csv(path, nrows=0).columns
dtypes = {c: "float64" for c in header if c not in str_cols}
dtypes.update({c: "string" for c in str_cols})

out_dir = Path(r"C:\Users\HP\Desktop\Ejust\Projects\Data Security\nids\data\processed")
out_dir.mkdir(exist_ok=True)

benign_only = {"train", "val"}
writers, row0 = {}, 0
for chunk in pd.read_csv(path, chunksize=500_000, dtype=dtypes):
    chunk.insert(0, "row_id", np.arange(row0, row0 + len(chunk)))
    for name, (s, e) in bounds.items():
        lo, hi = max(s, row0), min(e, row0 + len(chunk))
        if lo >= hi:
            continue
        part = chunk.iloc[lo - row0: hi - row0]
        if name in benign_only:
            part = part[part["Attack"] == "Benign"]
        if part.empty:
            continue
        table = pa.Table.from_pandas(part, preserve_index=False)
        if name not in writers:
            writers[name] = pq.ParquetWriter(out_dir / f"{name}.parquet", table.schema, compression="zstd")
        writers[name].write_table(table)
    row0 += len(chunk)

for w in writers.values():
    w.close()
print("done")

done


In [5]:
hosts = {}
for name in names:
    t = pd.read_parquet(out_dir / f"{name}.parquet", columns=["IPV4_SRC_ADDR", "Attack"])
    print(f"\n{name}: {len(t):,} rows")
    print(t["Attack"].value_counts().to_string())
    hosts[name] = t.loc[t["Attack"] == "Benign", "IPV4_SRC_ADDR"].value_counts()
    print("benign hosts with >=100 flows:", int((hosts[name] >= 100).sum()))

train_h = set(hosts["train"][hosts["train"] >= 100].index)
test_h = set(hosts["test"][hosts["test"] >= 100].index)
print("\ntest hosts (>=100 benign) also in train:", len(test_h & train_h), "of", len(test_h))


train: 3,659,681 rows
Attack
Benign    3659681
benign hosts with >=100 flows: 23

val: 539,110 rows
Attack
Benign    539110
benign hosts with >=100 flows: 19

tune: 894,775 rows
Attack
Benign       510816
scanning     363461
password       7138
injection      4602
ddos           4284
dos            2537
xss            1874
mitm             63
benign hosts with >=100 flows: 20

test: 8,182,234 rows
Attack
xss           2435788
ddos          2005385
Benign        1130384
password      1119916
dos            682153
injection      671296
scanning       110293
backdoor        16809
mitm             6818
ransomware       3392
benign hosts with >=100 flows: 34

test hosts (>=100 benign) also in train: 20 of 34


In [6]:
test = pd.read_parquet(out_dir / "test.parquet", columns=["row_id", "IPV4_SRC_ADDR", "Attack"])
print(test["Attack"].value_counts().to_string())

train_hosts = set(pd.read_parquet(out_dir / "train.parquet", columns=["IPV4_SRC_ADDR"])["IPV4_SRC_ADDR"])
tb = test[test["Attack"] == "Benign"]
seen = tb["IPV4_SRC_ADDR"].isin(train_hosts)

print("\ntest benign flows:", len(tb))
print("share of test benign flows from hosts seen in train:", round(seen.mean(), 3))
print("unseen benign hosts with the most flows:")
print(tb.loc[~seen, "IPV4_SRC_ADDR"].value_counts().head(5))

Attack
xss           2435788
ddos          2005385
Benign        1130384
password      1119916
dos            682153
injection      671296
scanning       110293
backdoor        16809
mitm             6818
ransomware       3392

test benign flows: 1130384
share of test benign flows from hosts seen in train: 0.961
unseen benign hosts with the most flows:
IPV4_SRC_ADDR
0.0.0.0           5757
176.28.50.165     4258
52.59.177.21      3057
18.184.104.180    2644
18.194.169.124    1710
Name: count, dtype: int64[pyarrow]
